In [15]:
import pandas as pd
import numpy as np

# 1. Load the datasets directly from the Excel file
file_name = "/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2026-02-02_taxa_abundances_bacteria_20260202 (1).xlsx"

# Read the taxa percentage tab. 
# Based on your file, we skip the first row (header=1) to use the actual column names.
# (If your tab is named exactly "taxa percentage", change sheet_name='%' to sheet_name='taxa percentage')
df_taxa = pd.read_excel(file_name, sheet_name='%', header=1)

# Read the references tab
df_refs = pd.read_excel(file_name, sheet_name='references')

# 2. Prepare and Transpose the Taxa Profile
# Set 'taxa [%]' as the index so it becomes our column names after transposing
df_taxa = df_taxa.set_index('taxa [%]')

# Extract only the sample columns (DK7000, DK7001, etc.)
dk_cols = [col for col in df_taxa.columns if str(col).startswith('DK')]
df_profile = df_taxa[dk_cols].T  # Transpose: Rows = Samples, Columns = Taxa

initial_columns = df_profile.shape[1]
print(f"Initial number of taxa (columns): {initial_columns}")

# 3. Filter 1: The Prevalence Filter
# Drop taxa where the value is 0 for MORE THAN 4 rows.
# This means we keep taxa where the count of 0s is <= 4.
zero_counts = (df_profile == 0).sum(axis=0)
df_filtered_1 = df_profile.loc[:, zero_counts <= 4]

# 4. Filter 2: The Abundance Threshold (Statistical Noise Filter)
# Calculate the mean abundance for each remaining taxa across all samples
mean_abundances = df_filtered_1.mean(axis=0)

# Define the threshold: 90th percentile of the mean abundances.
threshold = np.percentile(mean_abundances, 75) 
print(f"Calculated Abundance Threshold (90th percentile): {threshold:.5f}%")

# Keep only taxa whose mean abundance is strictly greater than the threshold
df_filtered_2 = df_filtered_1.loc[:, mean_abundances > threshold]

final_taxa_count = df_filtered_2.shape[1]

# 5. Merge Metadata (References)
# Prepare the references dataframe: keep only what we need and set the sample code as index
df_refs_subset = df_refs[['samples', 'Season', 'treatment', 'sampling time']].set_index('samples')

# Join the metadata to our filtered profile based on the DK index
df_final = df_filtered_2.join(df_refs_subset, how='left')

# Move the new metadata columns to the front for better readability
meta_cols = ['Season', 'treatment', 'sampling time']
taxa_cols = [col for col in df_final.columns if col not in meta_cols]
df_final = df_final[meta_cols + taxa_cols]

# 6. Save and Output Stats
#df_final.to_csv("cleaned_rice_V2.csv")

total_final_columns = df_final.shape[1]
print(f"Final number of taxa (after Filter 1 & 2): {final_taxa_count}")
print(f"Total columns in final dataset (Taxa + 3 Metadata): {total_final_columns}")
print("File successfully saved as 'cleaned_rice_V2.csv'")

Initial number of taxa (columns): 979
Calculated Abundance Threshold (90th percentile): 0.15802%
Final number of taxa (after Filter 1 & 2): 108
Total columns in final dataset (Taxa + 3 Metadata): 111
File successfully saved as 'cleaned_rice_V2.csv'


In [12]:
df_final

,Season,treatment,sampling time,Nitrosocosmicus oleophilus,Nitrosocosmicus sp.,Nitrososphaera sp.,Methanoperedens sp.,Bryobacter sp.,Paludibaculum sp.,Solibacter sp.,...,Methylobacter luteus,Azotobacter chroococcum,Azotobacter tropicalis,Pseudomonas peli,Povalibacter uvarum,Steroidobacter sp.,Treponema sp.,Udaeobacter sp.,Xiphinematobacter sp.,Opitutus sp.
DK7000,2,control S2,pre-planting,0.087197,2.432786,4.894637,0.061038,0.613283,0.093010,0.502834,...,0.061038,23.609940,2.985031,0.363319,0.212178,0.267403,0.066851,0.058131,0.081384,0.061038
DK7001,2,BH S2,pre-planting,0.408800,6.577969,10.955850,0.193251,2.058867,0.118924,1.018285,...,0.089193,0.000000,0.059462,0.040880,0.583470,0.769288,0.037164,0.107775,0.356771,0.252713
DK7002,2,KH S2,pre-planting,0.242708,7.298884,13.348925,0.172102,1.906359,0.194166,0.895812,...,0.167689,0.088257,0.114735,0.048542,0.511893,0.829619,0.057367,0.123560,0.264772,0.127973
DK7003,1,Control S1,harvest,0.246596,5.161156,8.116838,0.382051,1.601139,0.364685,0.840511,...,0.093776,0.055571,0.166713,0.100722,0.336899,0.555710,3.025146,0.145874,0.260489,0.236177
DK7004,1,KH S1,harvest,0.027903,4.899827,10.943691,0.373905,1.685362,0.491099,1.791395,...,0.669680,0.424131,0.044645,0.000000,0.558067,0.731068,0.061387,0.139517,0.295775,0.156259
DK7005,2,control S2,harvest,0.239655,10.511445,11.835673,0.559648,3.544438,0.550797,2.350931,...,0.234208,1.715710,1.211209,0.196762,0.514713,0.723730,0.311143,0.635902,0.106211,0.348589
DK7006,2,BH S2,harvest,0.091089,5.333738,6.376196,0.430140,2.378422,0.485805,1.599109,...,0.172056,9.103790,3.339912,2.550478,0.425080,0.561712,0.789434,0.399777,0.055665,0.202419
DK7007,2,KH S2,harvest,0.064627,6.673369,7.587726,0.215424,1.744938,0.301594,0.988559,...,0.067021,28.766336,4.830293,0.117287,0.289626,0.418881,0.009574,0.292020,0.028723,0.141223


In [10]:
microbe_list = pd.DataFrame(df_final.columns, columns=['Microbe_Name'])
microbe_list.to_csv('df1_final_columns_v2.csv', index=False)
print("Saved all column names to 'df1_final_columns_v2.csv'")

Saved all column names to 'df1_final_columns_v2.csv'


In [16]:
import pandas as pd
import json

# 1. The JSON dictionary mapping
taxa_mapping = {
  "Ammonia_Oxidizers_AOA": ["Nitrosocosmicus sp.", "Nitrososphaera sp."],
  "Nitrite_Oxidizers_NOB": ["Nitrospira japonica", "Nitrospira sp."],
  "Azotobacter_and_Free_Living_N_Fixers": ["Azotobacter chroococcum", "Azotobacter tropicalis", "Clostridium sp.", "Neobacillus niacini"],
  "Azospirillum_and_Associative_N_Fixers": ["Azospira restricta", "Azovibrio sp.", "Azoarcus tolulyticus"],
  "Rhizobium_and_Symbiotic_N_Fixers": ["Microvirga sp.", "Hyphomicrobium sp.", "Rhodoplanes sp.", "Pedomicrobium sp.", "Reyranella sp."],
  "Pseudomonas_and_PGPR": ["Pseudomonas peli", "Sphingomonas sp."],
  "Actinomycetes_Organic_Decomposers": ["Ilumatobacter sp.", "Luedemannella sp.", "Micromonospora sp.", "Nocardioides sp.", "Haloactinopolyspora sp.", "Crossiella sp.", "Rubrobacter sp.", "Gaiella sp.", "Solirubrobacter sp."],
  "Acidobacteria": ["Bryobacter sp.", "Solibacter sp.", "Blastocatella sp.", "Luteitalea sp."],
  "Myxobacteria": ["Anaeromyxobacter sp.", "Haliangium sp."],
  "Planctomycetes": ["Gemmata sp.", "Pirellula sp."],
  "Other_Proteobacteria": ["Defluviicoccus sp.", "Aquabacterium limnoticum", "Hydrogenophaga sp.", "Ramlibacter ginsenosidimutans", "Povalibacter uvarum", "Steroidobacter sp."],
  "Anaerobic_Reducers_and_Spirochaetes": ["Geoalkalibacter sp.", "Treponema sp."]
}

# 2. Flatten mapping to quickly look up category by taxa name
taxa_to_category = {}
for category, taxa_list in taxa_mapping.items():
    for taxa in taxa_list:
        taxa_to_category[taxa] = category

# 3. Load your updated dataset
# index_col=0 ensures DK7000, DK7001 stay correctly set as the index
df = pd.read_csv("/Users/pawanpahune/AI_Pipeline_Metagenomics/rice_project/cleaned_rice_V2.csv", index_col=0)

# 4. Generate the new column names
new_column_names = {}
for col in df.columns:
    if col in taxa_to_category:
        # Append the category to the end of the taxa name
        new_name = f"{col}_{taxa_to_category[col]}"
        new_column_names[col] = new_name
    else:
        # Leave non-taxa columns (Season, treatment, sampling time) untouched
        new_column_names[col] = col

# 5. Apply the new names and save
df.rename(columns=new_column_names, inplace=True)
df.to_csv("analysis_v1.csv")

# 6. Verify Output
print(f"Renamed {len([col for col in new_column_names if col in taxa_to_category])} taxa columns.")
print("Saved successfully as 'analysis_v1.csv'")

Renamed 43 taxa columns.
Saved successfully as 'analysis_v1.csv'
